# LLM 급식 메뉴 추천 RAG 프로젝트

## 1. 데이터 로드

급식 메뉴, 메뉴별 식재료, 식품 영양성분 데이터를 결합하여
LLM/RAG 검색에 사용할 메뉴 데이터셋을 구축한다.

## 0. 실행 환경 설정
공통 라이브러리, OpenAI client, 프로젝트 경로를 한 번만 설정한다.
이후 셀에서는 같은 라이브러리를 다시 import하지 않는다.


In [ ]:
import json
import re
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [ ]:
# 현재 프로젝트 위치
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"

print("현재 작업 폴더 :", BASE_DIR)
print("dataset 경로  :", DATA_DIR)
print("dataset 존재  :", DATA_DIR.exists())

print("\n=== dataset 파일 목록 ===")

for file in DATA_DIR.iterdir():
    print(f"{file.name:40} {file.stat().st_size:,} bytes")

# Part 1. 원천 데이터 로드 및 구조 검증
세 원천 데이터(Menu master, ingredient, nutrition DB)를 불러오고 키·컬럼·시트 구조를 확인한다.


## 2. 메뉴 마스터 데이터 확인

MenuGen 메뉴 마스터 데이터를 불러오고
메뉴 수, 컬럼 구조, 결측치 여부를 확인한다.

In [ ]:
menu_path = DATA_DIR / "menugen_menu_master.csv"

menu_df = pd.read_csv(menu_path)

print("menu_df shape :", menu_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(menu_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(menu_df.head())

print("\n=== 결측치 개수 ===")
display(menu_df.isnull().sum())

## 3. 메뉴별 식재료 데이터 확인

메뉴 코드(`fd_Code`)를 기준으로 각 메뉴에 연결된 식재료 데이터를 불러오고,
행 수, 컬럼 구조, 결측치를 확인한다.

In [ ]:
ingredient_path = DATA_DIR / "menugen_menu_ingredients.csv"

ingredient_df = pd.read_csv(ingredient_path)

print("ingredient_df shape :", ingredient_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(ingredient_df.columns):
    print(i, col)

print("\n=== 상위 10행 ===")
display(ingredient_df.head(10))

print("\n=== 결측치 개수 ===")
display(ingredient_df.isnull().sum())

print("\n=== 고유 메뉴 코드 수 ===")
print(ingredient_df["menu_fd_Code"].nunique())

## 4. 메뉴 마스터와 식재료 데이터 연결 검증

메뉴 코드 기준으로 두 데이터셋이 정상적으로 대응되는지 확인하고,
식재료가 없는 메뉴 또는 메뉴 마스터에 존재하지 않는 코드가 있는지 검증한다.

In [ ]:
# 메뉴 마스터 코드
menu_codes = set(menu_df["fd_Code"])

# 식재료 데이터의 메뉴 코드
ingredient_menu_codes = set(ingredient_df["menu_fd_Code"])

print("메뉴 마스터 고유 코드 수 :", len(menu_codes))
print("식재료 데이터 고유 코드 수 :", len(ingredient_menu_codes))

# 메뉴 마스터에는 있지만 식재료 데이터에는 없는 메뉴
missing_ingredient_codes = menu_codes - ingredient_menu_codes

# 식재료 데이터에는 있지만 메뉴 마스터에는 없는 메뉴
unknown_menu_codes = ingredient_menu_codes - menu_codes

print("\n=== 식재료 데이터가 없는 메뉴 ===")
print("개수 :", len(missing_ingredient_codes))
print(list(missing_ingredient_codes)[:20])

print("\n=== 메뉴 마스터에 존재하지 않는 식재료 메뉴 코드 ===")
print("개수 :", len(unknown_menu_codes))
print(list(unknown_menu_codes)[:20])

# 메뉴별 식재료 개수 확인
ingredient_count = (
    ingredient_df
    .groupby("menu_fd_Code")
    .size()
    .sort_values(ascending=False)
)

print("\n=== 메뉴별 식재료 개수 통계 ===")
display(ingredient_count.describe())

print("\n=== 식재료가 가장 많은 메뉴 Top 10 ===")
display(ingredient_count.head(10))

## 5. 국가표준식품성분 데이터 구조 확인

영양성분 엑셀 파일의 시트 목록과 각 시트의 크기를 확인하여
실제 데이터가 존재하는 시트를 식별한다.

In [ ]:
nutrition_path = DATA_DIR / "food_nutrition.xlsx"

# 엑셀 파일의 시트 목록 확인
xls = pd.ExcelFile(nutrition_path)

print("=== 시트 목록 ===")
print(xls.sheet_names)

print("\n=== 시트별 데이터 크기 ===")

for sheet in xls.sheet_names:
    temp_df = pd.read_excel(
        nutrition_path,
        sheet_name=sheet
    )
    
    print(f"{sheet} : {temp_df.shape}")

## 6. 국가표준식품성분 Database 10.4 로드

국가표준식품성분 데이터 중 최신 버전인 10.4를 기준 데이터로 사용한다.
이후 MenuGen 식재료 데이터와 연결하기 위해 식품코드, 식품명 및 주요 영양성분 컬럼을 확인한다.

In [ ]:
nutrition_df = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4"
)

print("nutrition_df shape :", nutrition_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(nutrition_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(nutrition_df.head())

print("\n=== 결측치 개수 상위 30개 ===")
display(
    nutrition_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

## 7. 국가표준식품성분 DB 10.4 헤더 정제

엑셀의 실제 컬럼명은 두 번째 행에 존재하며,
그 아래 단위 행을 제거하여 실제 식품 데이터만 구성한다.

In [ ]:
# DB10.4 원본 로드
nutrition_raw = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4",
    header=None
)

# 헤더 구성 정보
upper_header = nutrition_raw.iloc[0]
detail_header = nutrition_raw.iloc[1]
unit_row = nutrition_raw.iloc[2].copy()

# 세부 컬럼명이 있으면 사용하고,
# 비어 있으면 상위분류명을 컬럼명으로 사용
column_names = detail_header.copy()

column_names = column_names.where(
    column_names.notna(),
    upper_header
)


# 컬럼명 정규화 함수
def clean_column_names(columns):
    return (
        pd.Index(columns)
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


column_names = clean_column_names(column_names)

# 실제 식품 데이터
nutrition_df = (
    nutrition_raw
    .iloc[3:]
    .reset_index(drop=True)
)

nutrition_df.columns = column_names

# 단위 정보의 인덱스도 동일한 컬럼명으로 연결
unit_row.index = column_names


print("nutrition_df shape :", nutrition_df.shape)
display(nutrition_df.head(2))

# Part 2. 영양 DB 전처리
영양값 표기 규칙을 수치형으로 변환하고, 결측·중복·이상치·커버리지를 확인한 뒤 추천에 사용할 영양 Feature를 선별한다.


## 8. DB10.4 영양성분 표기 규칙

국가표준식품성분 DB10.4 설명문의 표기 기준을 적용한다.

- `-` : 측정되지 않은(unmeasured) 결측값
- `Tr` : 검출되었으나 정량 가능한 최소 농도 이하의 미량(trace)
- `(수치)` : 인용되었거나 재료량을 이용해 환산한 수치
- `(Tr)` : 괄호 표기가 적용된 미량값

원본값은 보존하고 계산용 수치와 데이터 상태를 별도로 관리한다.

In [ ]:
def parse_nutrition_value(value):
    if pd.isna(value):
        return np.nan, "missing"

    value = str(value).strip()

    if value == "-":
        return np.nan, "unmeasured"

    if value.lower() == "tr":
        return np.nan, "trace"

    if value.lower() == "(tr)":
        return np.nan, "quoted_or_converted_trace"

    if re.fullmatch(r"\([\d,]+(?:\.\d+)?\)", value):
        numeric_value = value[1:-1].replace(",", "")
        return float(numeric_value), "quoted_or_converted"

    numeric_value = pd.to_numeric(
        value.replace(",", ""),
        errors="coerce"
    )

    if pd.notna(numeric_value):
        return float(numeric_value), "numeric"

    return np.nan, "unknown"

## 9. 영양성분 컬럼 그룹 구성

DB10.4의 상위 헤더 정보를 이용하여 전체 컬럼을 영양성분 그룹별로 분류한다.

전체 컬럼을 개별적으로 확인하지 않고,
영양성분 그룹 단위로 활용 범위를 먼저 선정한 뒤
필요한 세부 컬럼만 추출한다.

In [ ]:
# DB10.4 상위 헤더를 각 컬럼에 전달
group_names = nutrition_raw.iloc[0].ffill()

# 상위 헤더명 정규화
group_names = clean_column_names(group_names)

# 컬럼 메타데이터 구성
nutrition_meta_df = pd.DataFrame({
    "column": nutrition_df.columns,
    "group": group_names,
    "unit": unit_row.values
})

# 식품 기본정보는 별도 그룹으로 지정
info_cols = [
    "DB10.4 색인",
    "10개정 책자 색인",
    "식품군",
    "식품명",
    "출처"
]

nutrition_meta_df.loc[
    nutrition_meta_df["column"].isin(info_cols),
    "group"
] = "식품정보"


# 단위가 있는 컬럼 = 수치형 전처리 대상
# 영양성분뿐 아니라 식염상당량, 폐기율도 포함
value_cols = (
    nutrition_meta_df
    .loc[nutrition_meta_df["unit"].notna(), "column"]
    .tolist()
)


print("식품 정보 컬럼 수 :", len(info_cols))
print("수치형 전처리 대상 :", len(value_cols))


# 그룹별 구조 확인
group_summary_df = (
    nutrition_meta_df
    .groupby("group", dropna=False)
    .agg(
        컬럼수=("column", "count"),
        컬럼목록=("column", list)
    )
    .reset_index()
)

display(group_summary_df)

## 10. 수치형 컬럼 전체 정제

In [ ]:
# 원본 보존
nutrition_clean_df = nutrition_df.copy()

# 원본 값의 상태정보 저장
nutrition_status_df = pd.DataFrame(
    index=nutrition_df.index
)


for col in value_cols:
    parsed = nutrition_df[col].map(parse_nutrition_value)

    # 계산에 사용할 숫자값
    nutrition_clean_df[col] = parsed.map(
        lambda x: x[0]
    )

    # 값의 원래 상태
    nutrition_status_df[col] = parsed.map(
        lambda x: x[1]
    )


print("전처리 대상 컬럼 :", len(value_cols))
print("정제 데이터 shape :", nutrition_clean_df.shape)


print("\n=== 값 상태 집계 ===")

status_counts = (
    nutrition_status_df
    .stack()
    .value_counts()
)

display(status_counts)

## 결측치 

In [ ]:
missing_report = pd.DataFrame({
    "column": value_cols,
    "missing_count": [
        nutrition_clean_df[col].isna().sum()
        for col in value_cols
    ]
})

missing_report["missing_rate"] = (
    missing_report["missing_count"]
    / len(nutrition_clean_df)
    * 100
).round(2)

missing_report = missing_report.sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

display(missing_report)

## 중복 검증

In [ ]:
print("전체 식품 수 :", len(nutrition_clean_df))

print(
    "DB10.4 색인 중복 :",
    nutrition_clean_df["DB10.4 색인"].duplicated().sum()
)

print(
    "완전 중복 행 :",
    nutrition_clean_df.duplicated().sum()
)

print(
    "식품명 중복 :",
    nutrition_clean_df["식품명"].duplicated().sum()
)

## 이상치

In [ ]:
outlier_rows = []

for col in value_cols:
    series = nutrition_clean_df[col].dropna()

    if series.empty:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    outlier_count = (
        (series < lower) |
        (series > upper)
    ).sum()

    outlier_rows.append({
        "column": col,
        "min": series.min(),
        "max": series.max(),
        "Q1": q1,
        "Q3": q3,
        "IQR_outlier_count": outlier_count
    })

outlier_report = pd.DataFrame(outlier_rows)

display(
    outlier_report.sort_values(
        "IQR_outlier_count",
        ascending=False
    )
)

In [ ]:
# 모든 수치형 컬럼의 음수값 확인
negative_report = pd.DataFrame({
    "column": value_cols,
    "negative_count": [
        (nutrition_clean_df[col] < 0).sum()
        for col in value_cols
    ]
})

negative_report = negative_report[
    negative_report["negative_count"] > 0
]

print("=== 음수값 존재 컬럼 ===")
display(negative_report)


# 가식부 100g 기준 물리적 범위 검사
physical_rows = []

for _, row in nutrition_meta_df.iterrows():

    col = row["column"]
    unit = row["unit"]

    if col not in value_cols:
        continue

    series = nutrition_clean_df[col]

    # g / 100g 또는 % 값은 0~100 범위
    if unit in ["g", "%"]:
        lower = 0
        upper = 100

    # 에너지는 별도 범위
    elif col == "에너지":
        lower = 0
        upper = 1000

    else:
        continue

    invalid_mask = (
        (series < lower) |
        (series > upper)
    )

    physical_rows.append({
        "column": col,
        "unit": unit,
        "lower_bound": lower,
        "upper_bound": upper,
        "invalid_count": invalid_mask.sum(),
        "min": series.min(),
        "max": series.max()
    })


physical_outlier_report = pd.DataFrame(physical_rows)

print("\n=== 물리적 범위 이상값 ===")

display(
    physical_outlier_report.sort_values(
        "invalid_count",
        ascending=False
    )
)

## 컬럼 선별

In [ ]:
# 공식 총량 컬럼 확인
total_cols_df = nutrition_meta_df[
    nutrition_meta_df["column"].str.startswith("총 ", na=False)
][
    ["group", "column", "unit"]
].reset_index(drop=True)

display(total_cols_df)

In [ ]:
total_summary_df = (
    nutrition_meta_df
    .groupby("group")
    .agg(
        전체컬럼수=("column", "count"),
        총량컬럼=("column", lambda x: [
            col for col in x
            if str(col).startswith("총 ")
        ])
    )
    .reset_index()
)

total_summary_df["총량존재"] = (
    total_summary_df["총량컬럼"]
    .apply(len)
    .gt(0)
)

display(total_summary_df)

In [ ]:
# 수치형 컬럼 결측률 분포 확인
missing_stats = missing_report["missing_rate"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
)

display(missing_stats)

In [ ]:
q1 = missing_report["missing_rate"].quantile(0.25)
q3 = missing_report["missing_rate"].quantile(0.75)

iqr = q3 - q1

missing_threshold = q3 + 1.5 * iqr

print("Q1 :", round(q1, 2))
print("Q3 :", round(q3, 2))
print("IQR :", round(iqr, 2))
print("고결측 컬럼 기준 :", round(missing_threshold, 2), "%")


high_missing_cols = missing_report[
    missing_report["missing_rate"] > missing_threshold
].copy()

display(high_missing_cols)

In [ ]:
coverage_rows = []

for col in value_cols:
    series = nutrition_clean_df[col]

    total = len(series)
    missing_count = series.isna().sum()
    zero_count = (series == 0).sum()
    positive_count = (series > 0).sum()

    coverage_rows.append({
        "column": col,
        "group": nutrition_meta_df.loc[
            nutrition_meta_df["column"] == col,
            "group"
        ].iloc[0],

        "missing_rate": round(missing_count / total * 100, 2),
        "zero_rate": round(zero_count / total * 100, 2),
        "positive_rate": round(positive_count / total * 100, 2),
        "available_rate": round(
            (total - missing_count) / total * 100,
            2
        )
    })


coverage_report = pd.DataFrame(coverage_rows)

display(
    coverage_report.sort_values(
        "positive_rate"
    )
)

In [ ]:
# positive_rate 분포 확인
positive_stats = coverage_report["positive_rate"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
)

display(positive_stats)


# 하위 25% 경계를 저커버리지 후보 기준으로 사용
positive_threshold = coverage_report[
    "positive_rate"
].quantile(0.25)

print(
    "저커버리지 후보 기준 :",
    round(positive_threshold, 2),
    "%"
)


low_coverage_cols = coverage_report[
    coverage_report["positive_rate"] <= positive_threshold
].sort_values("positive_rate")

display(low_coverage_cols)

In [ ]:
# 데이터 분포에서 기준 자동 산출
missing_q3 = coverage_report[
    "missing_rate"
].quantile(0.75)

positive_q1 = coverage_report[
    "positive_rate"
].quantile(0.25)


quality_exclude_candidates = coverage_report[
    (coverage_report["missing_rate"] >= missing_q3)
    &
    (coverage_report["positive_rate"] <= positive_q1)
].sort_values(
    ["positive_rate", "missing_rate"],
    ascending=[True, False]
)


print("결측률 상위 25% 기준 :", round(missing_q3, 2), "%")
print("양수값 하위 25% 기준 :", round(positive_q1, 2), "%")
print("제외 후보 컬럼 수 :", len(quality_exclude_candidates))

display(quality_exclude_candidates)

In [ ]:
# 1차 Feature Selection
# 세부 아미노산/지방산은 대표 지표만 유지하고,
# 데이터 품질이 낮은 컬럼 제외

detail_groups = [
    "아미노산 Amino acids",
    "지방산 Fatty acids"
]

# 아미노산 대표 지표
amino_keep_cols = [
    "총 아미노산",
    "총 필수 아미노산"
]

# 지방산 대표 지표
fatty_keep_cols = [
    "총 필수 지방산",
    "총 포화 지방산",
    "총 불포화 지방산",
    "오메가3 지방산",
    "오메가6 지방산",
    "총 트랜스 지방산"
]

# 아미노산/지방산 외 그룹은 우선 유지
base_cols = nutrition_meta_df.loc[
    ~nutrition_meta_df["group"].isin(detail_groups),
    "column"
].tolist()

# 실제 DB에 존재하는 대표 컬럼만 추가
summary_cols = [
    col
    for col in amino_keep_cols + fatty_keep_cols
    if col in nutrition_clean_df.columns
]

# 데이터 품질 기준 제외
quality_exclude_cols = set(
    quality_exclude_candidates["column"]
)

selected_cols = [
    col
    for col in base_cols + summary_cols
    if col not in quality_exclude_cols
]

nutrition_selected_df = nutrition_clean_df[
    selected_cols
].copy()

print("전체 컬럼 :", nutrition_clean_df.shape[1])
print("1차 선별 컬럼 :", nutrition_selected_df.shape[1])

In [ ]:
# LLM 급식 추천용 핵심 영양 지표
# 대상: 유아·학생·성인·노인·질환별 식단 조건을 폭넓게 고려

llm_feature_cols = [
    # 식품 정보
    "DB10.4 색인",
    "식품군",
    "식품명",
    "출처",

    # 기본 영양 / 체중 / 당뇨 / 성장 / 영양불량
    "에너지",
    "수분",
    "단백질",
    "지방",
    "탄수화물",
    "당류",
    "총 식이섬유",

    # 비타민
    "비타민 A",       # 성장, 시각, 학교급식
    "비타민 D",       # 성장, 골건강, 노인
    "비타민 C",       # 학교급식, 철 흡수 등
    "티아민",          # 비타민 B1, 학교급식
    "리보플라빈",      # 비타민 B2, 학교급식
    # "엽산",            # 임신, 조혈
    "비타민 B12",     # 조혈, 노인
    "비타민 K1",      # 혈액응고 관련 특수 식단

    # 무기질
    "칼슘",            # 성장, 골건강
    "철",              # 성장, 빈혈
    "마그네슘",        # 골·근육
    "인",              # 골건강 + 신장질환
    "칼륨",            # 혈압 + 신장질환
    "나트륨",          # 저염, 고혈압, 신장질환
    "아연",            # 성장, 면역, 회복
    "요오드",          # 갑상선

    # 단백질 질 보조 지표
    "총 필수 아미노산",

    # 지방의 질
    "총 포화 지방산",
    "총 불포화 지방산",
    "오메가3 지방산",
    "오메가6 지방산",
    "총 트랜스 지방산",

    # 사용자 질의 대응용 보조 지표
    "콜레스테롤"
]

# 1차 선별된 67개 중에서만 선택
final_cols = [
    col for col in llm_feature_cols
    if col in nutrition_selected_df.columns
]

# 혹시 1차 전처리에서 이미 빠졌거나
# DB 이름이 다른 컬럼 확인
missing_target_cols = [
    col for col in llm_feature_cols
    if col not in nutrition_selected_df.columns
]

# LLM용 최종 영양 데이터
llm_nutrition_df = nutrition_selected_df[
    final_cols
].copy()

print("1차 선별 컬럼 :", nutrition_selected_df.shape[1])
print("LLM 최종 컬럼 :", llm_nutrition_df.shape[1])

print("\n최종 컬럼:")
print(final_cols)

print("\n찾지 못한 컬럼:")
print(missing_target_cols)

# Part 3. MenuGen ↔ 영양 DB 통합
식품명을 정규화해 식재료와 영양 DB를 연결하고, 100g 기준 영양값을 실제 사용 중량 기준으로 환산한 뒤 메뉴 단위로 집계한다.


## 식품명 정규화

In [ ]:
def normalize_food_name(name):
    if pd.isna(name):
        return ""

    name = str(name).strip()

    # 전각 쉼표 등 기본 표기 통일
    name = name.replace("，", ",")

    # 띄어쓰기 차이 제거
    name = re.sub(r"\s+", "", name)

    # 영문 포함 시 대소문자 차이 제거
    name = name.casefold()

    return name


# 원본은 그대로 보존하고 매칭용 컬럼만 추가
ingredient_df["food_name_norm"] = (
    ingredient_df["ingredient_food_Nm"]
    .map(normalize_food_name)
)

llm_nutrition_df = llm_nutrition_df.copy()

llm_nutrition_df["food_name_norm"] = (
    llm_nutrition_df["식품명"]
    .map(normalize_food_name)
)

print("MenuGen 식재료 행 :", len(ingredient_df))
print("MenuGen 고유 식재료명 :", ingredient_df["food_name_norm"].nunique())
print("영양 DB 식품 수 :", len(llm_nutrition_df))
print("영양 DB 정규화명 중복 :", llm_nutrition_df["food_name_norm"].duplicated().sum())

In [ ]:
# 식품정보를 제외한 실제 영양성분 컬럼
info_cols = [
    "DB10.4 색인",
    "식품군",
    "식품명",
    "출처"
]

nutrient_cols = [
    col for col in final_cols
    if col not in info_cols
]

# MenuGen 식재료 ↔ 영양 DB 연결
ingredient_nutrition_df = ingredient_df.merge(
    llm_nutrition_df[
        ["food_name_norm"] + nutrient_cols
    ],
    on="food_name_norm",
    how="left"
)

# 100g 기준 영양값 → 실제 사용 중량 기준으로 환산
for col in nutrient_cols:
    ingredient_nutrition_df[col] = (
        ingredient_nutrition_df[col]
        * ingredient_nutrition_df["ingredient_food_Wgh"]
        / 100
    )

In [ ]:
menu_nutrition_df = (
    ingredient_nutrition_df
    .groupby(
        ["menu_fd_Code", "menu_fd_Nm"],
        as_index=False
    )[nutrient_cols]
    .sum(min_count=1)
)

print("메뉴별 영양 데이터 :", menu_nutrition_df.shape)

display(menu_nutrition_df.head())

In [ ]:
# 메뉴별 알레르기 정보 집계
menu_allergy_df = (
    ingredient_df
    .groupby(
        ["menu_fd_Code", "menu_fd_Nm"],
        as_index=False
    )["ingredient_allrgy_Info"]
    .agg(
        lambda x: ", ".join(
            sorted(
                set(
                    str(v).strip()
                    for v in x
                    if pd.notna(v)
                    and str(v).strip()
                )
            )
        )
    )
)

# 컬럼명 변경
menu_allergy_df = menu_allergy_df.rename(
    columns={
        "ingredient_allrgy_Info": "allergy_info"
    }
)

print(
    "메뉴별 알레르기 데이터 :",
    menu_allergy_df.shape
)

display(menu_allergy_df.head())

In [ ]:
menu_rag_df = menu_nutrition_df.merge(
    menu_allergy_df,
    on=[
        "menu_fd_Code",
        "menu_fd_Nm"
    ],
    how="left"
)

print(
    "최종 메뉴 데이터 :",
    menu_rag_df.shape
)

display(
    menu_rag_df[
        [
            "menu_fd_Code",
            "menu_fd_Nm",
            "allergy_info"
        ]
    ].head()
)

In [ ]:
menu_meta_df = menu_df[
    [
        "fd_Code",
        "upper_Fd_Grupp_Nm",
        "fd_Grupp_Nm"
    ]
].copy()

menu_meta_df = menu_meta_df.rename(
    columns={
        "fd_Code": "menu_fd_Code"
    }
)

menu_rag_df = menu_rag_df.merge(
    menu_meta_df,
    on="menu_fd_Code",
    how="left"
)

print(
    menu_rag_df[
        [
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ].head()
)

In [ ]:
display(
    menu_rag_df[
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    )
)

In [ ]:
category_summary = (
    menu_rag_df
    .groupby("upper_Fd_Grupp_Nm")
    .agg(
        menu_count=("menu_fd_Code", "count")
    )
    .sort_index()
)

display(category_summary)

In [ ]:
# 추천 대상에서 제외할 명확한 대분류
NON_MENU_GROUPS = {
    "원재료",
    "장류",
    "주류"
}

# 분류상 메뉴군에 들어가 있지만 실제 추천 메뉴로는 부적절한 예외
EXCLUDED_MENU_NAMES = {
    "육수(소고기)"
}

recommendable_menu_df = (
    menu_rag_df[
        ~menu_rag_df["upper_Fd_Grupp_Nm"]
        .isin(NON_MENU_GROUPS)
        &
        ~menu_rag_df["menu_fd_Nm"]
        .isin(EXCLUDED_MENU_NAMES)
    ]
    .copy()
    .reset_index(drop=True)
)

print("전체 MenuGen :", len(menu_rag_df))
print("추천 가능 메뉴 :", len(recommendable_menu_df))

In [ ]:
problem_names = [
    "잣(볶은것)",
    "잣(생것)",
    "은행(볶은것)",
    "은행(생것)",
    "육수(소고기)"
]

remaining_problem_df = (
    recommendable_menu_df[
        recommendable_menu_df["menu_fd_Nm"]
        .isin(problem_names)
    ][
        [
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
)

display(remaining_problem_df)

print(
    "추천 후보에 남아있는 문제 항목 :",
    remaining_problem_df["menu_fd_Nm"].tolist()
)

In [ ]:
# 추천 후보 중 비메뉴성 항목 의심 목록 확인
# ※ 이 단계에서는 제거하지 않고 검토만 한다.

SUSPICIOUS_NAME_PATTERN = (
    r"육수|"
    r"생것|말린것|건조|"
    r"분말|가루|"
    r"소스|양념|"
    r"원액|농축액"
)

suspicious_menu_df = (
    recommendable_menu_df[
        recommendable_menu_df["menu_fd_Nm"]
        .str.contains(
            SUSPICIOUS_NAME_PATTERN,
            regex=True,
            na=False
        )
    ][
        [
            "menu_fd_Code",
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .sort_values(
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm",
            "menu_fd_Nm"
        ]
    )
)

print(
    "비메뉴성 의심 항목 수 :",
    len(suspicious_menu_df)
)

display(
    suspicious_menu_df
)

In [ ]:
# 어떤 키워드 때문에 의심 항목으로 잡혔는지 분류

SUSPICIOUS_KEYWORDS = [
    "육수",
    "생것",
    "말린것",
    "건조",
    "분말",
    "가루",
    "소스",
    "양념",
    "원액",
    "농축액"
]

def find_suspicious_reason(name):
    name = str(name)

    matched = [
        keyword
        for keyword in SUSPICIOUS_KEYWORDS
        if keyword in name
    ]

    return ", ".join(matched)


suspicious_menu_df = suspicious_menu_df.copy()

suspicious_menu_df["reason"] = (
    suspicious_menu_df["menu_fd_Nm"]
    .map(find_suspicious_reason)
)

display(
    suspicious_menu_df[
        [
            "reason",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .value_counts()
    .reset_index(name="count")
    .sort_values(
        ["reason", "count"],
        ascending=[True, False]
    )
)

# Part 4. 전처리 검증 및 시각화
아래 셀들은 추천 파이프라인의 필수 연산이 아니라, 데이터 품질과 전처리 결과를 설명하기 위한 분석/발표용 시각화다.


In [ ]:
# ============================================
# 영양성분 원본 값 상태 분포 시각화
# ============================================


mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

# status_counts를 그래프용 Series로 사용
plot_data = status_counts.sort_values(ascending=False)


plt.figure(figsize=(9, 5))

bars = plt.bar(
    plot_data.index.astype(str),
    plot_data.values
)

plt.title(
    "영양성분 원본 값 상태 분포",
    fontsize=15
)

plt.xlabel("값 상태")
plt.ylabel("건수")

plt.xticks(rotation=30)


# 막대 위 건수 표시
for bar, count in zip(bars, plot_data.values):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=10
    )


plt.tight_layout()
plt.show()


In [ ]:
special_status = status_counts.drop(
    labels=["numeric", "measured"],
    errors="ignore"
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    special_status.index.astype(str),
    special_status.values
)

plt.title("영양성분 특수값 상태 분포")
plt.xlabel("값 상태")
plt.ylabel("건수")

plt.xticks(rotation=30)

for bar, count in zip(bars, special_status.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 영양성분별 결측률 시각화
# ============================================


# 영양성분별 결측률 계산
missing_rate = (
    ingredient_nutrition_df[nutrient_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


print("=== 영양성분별 결측률 상위 15개 ===")
display(missing_rate.head(15))


# 결측률 상위 15개 시각화
plot_data = missing_rate.head(15).sort_values()

plt.figure(figsize=(10, 6))

bars = plt.barh(
    plot_data.index,
    plot_data.values
)

plt.title(
    "영양성분별 결측률 상위 15개",
    fontsize=15
)

plt.xlabel("결측률 (%)")
plt.ylabel("영양성분")


# 막대 끝에 결측률 표시
for bar, rate in zip(bars, plot_data.values):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}%",
        va="center",
        fontsize=10
    )


plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 메뉴별 주요 영양성분 상관관계 히트맵
# ============================================



# 주요 영양성분 자동 선택
keywords = [
    "에너지",
    "단백질",
    "지방",
    "탄수화물",
    "당류",
    "식이섬유",
    "나트륨",
    "칼슘",
    "철"
]

heatmap_cols = []

for keyword in keywords:
    matched = [
        col for col in menu_nutrition_df.columns
        if keyword in col
    ]

    if matched:
        heatmap_cols.append(matched[0])


print("히트맵 사용 컬럼:")
for col in heatmap_cols:
    print("-", col)


# 상관계수 계산
corr = (
    menu_nutrition_df[heatmap_cols]
    .corr()
)


# 시각화
plt.figure(figsize=(10, 8))

im = plt.imshow(
    corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    im,
    label="상관계수"
)

plt.xticks(
    range(len(heatmap_cols)),
    heatmap_cols,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(heatmap_cols)),
    heatmap_cols
)


# 셀 안에 상관계수 표시
for i in range(len(corr)):
    for j in range(len(corr)):

        plt.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )


plt.title(
    "메뉴별 주요 영양성분 상관관계",
    fontsize=15
)

plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


vif_cols = heatmap_cols

vif_df = (
    menu_nutrition_df[vif_cols]
    .dropna()
)

vif_result = pd.DataFrame({
    "영양성분": vif_cols,
    "VIF": [
        variance_inflation_factor(vif_df.values, i)
        for i in range(len(vif_cols))
    ]
})

vif_result = vif_result.sort_values(
    "VIF",
    ascending=False
)

display(vif_result)


In [ ]:
plt.figure(figsize=(10, 6))

plot_data = category_summary["menu_count"].sort_values()

bars = plt.barh(
    plot_data.index,
    plot_data.values
)

plt.title("메뉴 대분류별 데이터 분포")
plt.xlabel("메뉴 수")
plt.ylabel("대분류")

for bar, count in zip(bars, plot_data.values):
    plt.text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,}",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
before_count = len(menu_rag_df)
after_count = len(recommendable_menu_df)

plt.figure(figsize=(7, 5))

bars = plt.bar(
    ["전체 메뉴", "추천 가능 메뉴"],
    [before_count, after_count]
)

plt.title("추천 대상 필터링 전·후 메뉴 수")
plt.ylabel("메뉴 수")

for bar, count in zip(
    bars,
    [before_count, after_count]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f"{count:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
reason_counts = (
    suspicious_menu_df["reason"]
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(9, 5))

bars = plt.barh(
    reason_counts.index,
    reason_counts.values
)

plt.title("비메뉴성 의심 항목 키워드별 검출 건수")
plt.xlabel("검출 메뉴 수")

for bar, count in zip(
    bars,
    reason_counts.values
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        str(count),
        va="center"
    )

plt.tight_layout()
plt.show()

# Part 5. 사용자 요청 구조화
LLM이 자연어 요청을 target / allergies / nutrition_high / nutrition_low / diseases / keywords로 구조화한다.
알레르기 필터 함수도 이 단계에서 정의한다.


In [ ]:
def extract_user_conditions(user_query):

    response = client.responses.create(
        model="gpt-5.6-luna",
        reasoning={
            "effort": "low"
        },

        input=f"""
너는 급식 메뉴 추천 시스템의 조건 추출기다.

사용자의 요청에서 실제 메뉴 검색과 영양 랭킹에 필요한 조건만 추출해라.

규칙:
- target: 급식 대상
- allergies: 알레르기 목록
- nutrition_high: 많이 섭취하고 싶은 영양소
- nutrition_low: 적게 섭취하고 싶은 영양소
- diseases: 질환 또는 건강 상태
- keywords: 위 항목에 포함되지 않는 실제 추천 조건

'급식', '메뉴', '추천', '음식'처럼
추천 시스템 자체를 설명하는 일반적인 단어는 keywords에 넣지 마라.

사용자 요청:
{user_query}
""",

        text={
            "format": {
                "type": "json_schema",
                "name": "meal_conditions",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "target": {
                            "type": ["string", "null"]
                        },
                        "allergies": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "nutrition_high": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "nutrition_low": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "diseases": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "keywords": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "target",
                        "allergies",
                        "nutrition_high",
                        "nutrition_low",
                        "diseases",
                        "keywords"
                    ],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)

In [ ]:
test_query = """
고혈압 있는 어르신 급식이야.
대두 알레르기가 있고 나트륨은 낮고
단백질은 충분한 메뉴 추천해줘.
"""

conditions = extract_user_conditions(test_query)

print(json.dumps(
    conditions,
    ensure_ascii=False,
    indent=2
))

In [ ]:
def filter_allergies(df, allergies):
    result = df.copy()

    for allergy in allergies:
        mask = (
            result["allergy_info"]
            .fillna("")
            .str.contains(
                str(allergy),
                case=False,
                regex=False
            )
        )

        result = result[~mask]

    return result


def rank_menus(
    df,
    nutrition_high,
    nutrition_low,
    top_k=10
):
    result = df.copy()

    # 실제 데이터에 존재하는 영양소만 사용
    high_cols = [
        col for col in nutrition_high
        if col in result.columns
    ]

    low_cols = [
        col for col in nutrition_low
        if col in result.columns
    ]

    ranking_cols = list(
        dict.fromkeys(high_cols + low_cols)
    )

    # 요청된 영양소가 없으면 점수 계산 없이 반환
    if not ranking_cols:
        result["nutrition_score"] = 0.0
        return result.head(top_k)

    # 요청 영양소 값이 없는 메뉴는 랭킹에서 제외
    result = result.dropna(
        subset=ranking_cols
    ).copy()

    score_cols = []

    # 높을수록 좋은 영양소
    for col in high_cols:
        score_col = f"_high_{col}"

        result[score_col] = (
            result[col]
            .rank(pct=True)
        )

        score_cols.append(score_col)

    # 낮을수록 좋은 영양소
    for col in low_cols:
        score_col = f"_low_{col}"

        result[score_col] = (
            1
            - result[col].rank(pct=True)
        )

        score_cols.append(score_col)

    # 조건별 점수 평균
    result["nutrition_score"] = (
        result[score_cols]
        .mean(axis=1)
    )

    result = (
        result
        .sort_values(
            "nutrition_score",
            ascending=False
        )
        .head(top_k)
    )

    return result

# Part 6. 메뉴 RAG 인덱스 구축
추천 가능 메뉴를 문서 형태로 변환하고 BGE-M3 임베딩 + FAISS cosine 검색 인덱스를 구축한다.


## 메뉴를 RAG Document로 변환

In [ ]:
# 메뉴별 식재료 목록 생성
def join_unique_ingredients(values):
    return ", ".join(
        dict.fromkeys(
            str(v).strip()
            for v in values
            if pd.notna(v) and str(v).strip()
        )
    )


ingredient_text_map = (
    ingredient_df
    .groupby("menu_fd_Code")["ingredient_food_Nm"]
    .agg(join_unique_ingredients)
    .to_dict()
)


# 추천 가능한 메뉴 데이터에 식재료 정보 추가
recommendable_menu_df = recommendable_menu_df.copy()

recommendable_menu_df["ingredient_text"] = (
    recommendable_menu_df["menu_fd_Code"]
    .map(ingredient_text_map)
    .fillna("")
)


def safe_text(value, default="없음"):
    if pd.isna(value):
        return default

    value = str(value).strip()

    return value if value else default


# BGE-M3 검색용 메뉴 문서 생성
def make_menu_document(row):

    lines = [
        f"메뉴명: {row['menu_fd_Nm']}",
        f"메뉴 대분류: {safe_text(row['upper_Fd_Grupp_Nm'])}",
        f"메뉴 소분류: {safe_text(row['fd_Grupp_Nm'])}",
        f"식재료: {safe_text(row['ingredient_text'])}"
    ]

    return "\n".join(lines)


menu_documents_df = recommendable_menu_df[
    [
        "menu_fd_Code",
        "menu_fd_Nm",
        "upper_Fd_Grupp_Nm",
        "fd_Grupp_Nm",
        "ingredient_text"
    ]
].copy()

menu_documents_df["document"] = (
    recommendable_menu_df.apply(
        make_menu_document,
        axis=1
    )
)

print("RAG 문서 수 :", len(menu_documents_df))
print()
print(menu_documents_df["document"].iloc[0])

In [ ]:
# BGE-M3 모델 로드
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import faiss

BGE_MODEL_NAME = "BAAI/bge-m3"

bge_model = BGEM3FlagModel(
    BGE_MODEL_NAME,
    use_fp16=False
)

print("BGE-M3 로드 완료")

In [ ]:
menu_texts = menu_documents_df[
    "document"
].tolist()

output = bge_model.encode(
    menu_texts,
    batch_size=8,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

menu_embeddings = np.asarray(
    output["dense_vecs"],
    dtype="float32"
)

print(
    "Embedding shape :",
    menu_embeddings.shape
)

In [ ]:
# RAG 임베딩 저장
rag_dir = Path("rag_store")
rag_dir.mkdir(exist_ok=True)

np.save(
    rag_dir / "bge_m3_menu_embeddings.npy",
    menu_embeddings
)

menu_documents_df.to_csv(
    rag_dir / "menu_documents.csv",
    index=False,
    encoding="utf-8-sig"
)

print("BGE-M3 임베딩 저장 완료")

In [ ]:
# 원본 embedding은 유지하고 FAISS용 복사본 사용
menu_vectors = menu_embeddings.copy()

# cosine similarity
faiss.normalize_L2(menu_vectors)

embedding_dim = menu_vectors.shape[1]

menu_index = faiss.IndexFlatIP(
    embedding_dim
)

menu_index.add(menu_vectors)

# FAISS 인덱스 저장
faiss.write_index(
    menu_index,
    str(rag_dir / "menu_index.faiss")
)

print("Embedding dimension :", embedding_dim)
print("FAISS 저장 문서 :", menu_index.ntotal)

In [ ]:
# BGE-M3 Retriever
def retrieve_menus(query, top_k=50):

    query_output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        query_output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scores, indices = menu_index.search(
        query_vector,
        top_k
    )

    retrieved = (
        recommendable_menu_df.iloc[indices[0]].copy()
    )

    retrieved["retrieval_score"] = scores[0]

    return retrieved

# Part 7. 식단 슬롯별 후보 검색 및 Gate
주식·국·주찬·부찬·김치 슬롯별로 후보를 검색하고, 알레르기 및 LLM Gate를 거쳐 최종 후보 Pool을 만든다.


In [ ]:
# 한 끼 급식 식단의 기본 구성
MEAL_SLOT_GROUPS = {
    "주식": {
        "밥류",
        "면 및 만두류",
        "죽류"
    },

    "국": {
        "국(탕)류",
        "찌개류"
    },

    "주찬": {
        "구이류",
        "볶음류",
        "조림류",
        "찜류",
        "전류",
        "튀김류"
    },

    "부찬": {
        "무침류",
        "볶음류",
        "조림류",
        "전류",
        "절임류"
    },

    "김치": {
        "김치류"
    }
}

In [ ]:
# 식단 슬롯별 FAISS Index 생성
slot_indexes = {}
slot_menu_dfs = {}

for slot_name, allowed_groups in MEAL_SLOT_GROUPS.items():

    # 해당 슬롯에 사용할 수 있는 메뉴 위치
    mask = (
        recommendable_menu_df[
            "upper_Fd_Grupp_Nm"
        ]
        .isin(allowed_groups)
        .to_numpy()
    )

    # 메뉴 dataframe
    slot_df = (
        recommendable_menu_df[
            mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    # 기존 BGE-M3 embedding에서 해당 행만 추출
    slot_vectors = (
        menu_embeddings[
            mask
        ]
        .copy()
    )

    faiss.normalize_L2(
        slot_vectors
    )

    slot_index = faiss.IndexFlatIP(
        slot_vectors.shape[1]
    )

    slot_index.add(
        slot_vectors
    )

    slot_menu_dfs[
        slot_name
    ] = slot_df

    slot_indexes[
        slot_name
    ] = slot_index

    print(
        slot_name,
        ":",
        len(slot_df)
    )

In [ ]:
def retrieve_slot_candidates(
    slot_name,
    semantic_query="",
    allergies=None,
    top_k=15
):
    if allergies is None:
        allergies = []

    slot_df = slot_menu_dfs[
        slot_name
    ]

    slot_index = slot_indexes[
        slot_name
    ]

    query = f"{slot_name} 급식 메뉴"

    if semantic_query:
        query += (
            "\n추가 조건: "
            + semantic_query
        )

    # Query Embedding
    query_output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        query_output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(
        query_vector
    )

    # 슬롯 내부 전체 후보를 유사도 순으로 검색
    search_k = slot_index.ntotal

    scores, indices = (
        slot_index.search(
            query_vector,
            search_k
        )
    )

    retrieved = (
        slot_df
        .iloc[indices[0]]
        .copy()
    )

    retrieved["retrieval_score"] = (
        scores[0]
    )

    # 알레르기 Hard Filter
    retrieved = filter_allergies(
        retrieved,
        allergies
    )

    # 안전 필터를 통과한 후보 중
    # Semantic similarity 상위 N개
    return (
        retrieved
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
semantic_parts = []

if conditions.get("target"):
    semantic_parts.append(
        conditions["target"]
    )

semantic_parts.extend(
    conditions.get(
        "keywords",
        []
    )
)

semantic_query = " ".join(
    semantic_parts
)

print("semantic_query :", semantic_query)

In [ ]:
PRE_GATE_POOL_SIZE = {
    "주식": 12,
    "국": 15,
    "주찬": 20,
    "부찬": 25,
    "김치": 10
}

slot_candidates = {}

for slot_name in MEAL_SLOT_GROUPS:

    slot_candidates[slot_name] = retrieve_slot_candidates(
        slot_name=slot_name,
        semantic_query=semantic_query,
        allergies=conditions["allergies"],
        top_k=PRE_GATE_POOL_SIZE[slot_name]
    )

    print(
        slot_name,
        ":",
        len(slot_candidates[slot_name])
    )

In [ ]:
slot_gate_items = []

for slot_name, df in slot_candidates.items():

    for idx, row in df.iterrows():

        slot_gate_items.append({
            "candidate_id": f"{slot_name}_{idx}",
            "slot": slot_name,
            "menu_name": row["menu_fd_Nm"],
            "upper_group": row["upper_Fd_Grupp_Nm"],
            "sub_group": row["fd_Grupp_Nm"]
        })

print(
    "검사할 슬롯 후보 수 :",
    len(slot_gate_items)
)

In [ ]:
def evaluate_slot_candidates(
    slot_gate_items,
    conditions
):

    target_context = " ".join([
        str(conditions.get("target") or ""),
        *conditions.get("keywords", [])
    ]).strip()

    response = client.responses.create(
        model="gpt-5.6-luna",
        reasoning={"effort": "low"},
        input=f"""
너는 급식 식단의 메뉴 역할 적합성을 판정하는 분류기다.

급식 대상:
{target_context}

각 후보가 현재 지정된 슬롯에 실제 급식 식단 구성상
자연스러운 경우에만 keep=true로 판단한다.

[슬롯 기준]

주식:
- 밥, 죽, 면 등 식사의 중심 탄수화물 음식
- 이유식처럼 현재 급식 대상과 명백히 맞지 않으면 제외

국:
- 국, 탕, 찌개 등 국물 음식
- 메뉴명이 국/탕류로 분류되었더라도
  통닭·백숙 등 독립적인 주찬 성격이 강하면 제외

주찬:
- 한 끼의 중심 반찬
- 육류, 생선, 달걀, 두부 등 단백질 중심 또는
  일반적으로 메인 반찬으로 제공되는 음식
- 간식, 디저트, 떡류, 일품식,
  채소 단독 소량 반찬은 제외

부찬:
- 주찬을 보조하는 일반적인 반찬
- 나물, 무침, 조림, 채소반찬 등
- 떡볶이·면·밥처럼 주식성 또는 일품식 성격이 강한 음식,
  간식·디저트는 제외

김치:
- 김치류만 유지

추가 규칙:
- 영양성분을 보고 판단하지 않는다.
- 질환 치료 적합성을 판단하지 않는다.
- 메뉴명과 분류를 근거로 식단 역할만 판단한다.
- 명백히 부자연스러운 슬롯 배치는 제외한다.
- 합리적으로 해당 역할에 제공될 수 있다면 유지한다.
- 메뉴명에 어린이, 유아, 영유아 등 특정 대상이 명시되어 있고 현재 급식 대상과 명백히 다르면 제외한다.

검사 대상:
{json.dumps(
    slot_gate_items,
    ensure_ascii=False,
    indent=2
)}
""",
        text={
            "format": {
                "type": "json_schema",
                "name": "slot_candidate_gate",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "candidate_id": {
                                        "type": "string"
                                    },
                                    "keep": {
                                        "type": "boolean"
                                    },
                                    "reason": {
                                        "type": "string"
                                    }
                                },
                                "required": [
                                    "candidate_id",
                                    "keep",
                                    "reason"
                                ],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(
        response.output_text
    )

In [ ]:
slot_gate_result = evaluate_slot_candidates(
    slot_gate_items,
    conditions
)

slot_gate_result

In [ ]:
rejected_candidates = [
    item
    for item in slot_gate_result["results"]
    if not item["keep"]
]

for item in rejected_candidates:
    print(
        item["candidate_id"],
        "→",
        item["reason"]
    )

In [ ]:
def apply_slot_gate(
    slot_candidates,
    slot_gate_result
):
    keep_map = {
        item["candidate_id"]: item["keep"]
        for item in slot_gate_result["results"]
    }

    filtered_slots = {}

    for slot_name, df in slot_candidates.items():

        keep_indices = []

        for idx in range(len(df)):

            candidate_id = (
                f"{slot_name}_{idx}"
            )

            if keep_map.get(
                candidate_id,
                True
            ):
                keep_indices.append(idx)

        filtered_slots[slot_name] = (
            df.iloc[keep_indices]
            .copy()
            .reset_index(drop=True)
        )

    return filtered_slots


filtered_slot_candidates = apply_slot_gate(
    slot_candidates,
    slot_gate_result
)

for slot_name, df in filtered_slot_candidates.items():
    print(
        slot_name,
        ":",
        len(df)
    )

In [ ]:
FINAL_POOL_SIZE = {
    "주식": 8,
    "국": 8,
    "주찬": 8,
    "부찬": 12,
    "김치": 8
}

for slot_name in filtered_slot_candidates:

    filtered_slot_candidates[slot_name] = (
        filtered_slot_candidates[slot_name]
        .head(FINAL_POOL_SIZE[slot_name])
        .reset_index(drop=True)
    )

    print(
        slot_name,
        ":",
        len(filtered_slot_candidates[slot_name])
    )

# Part 8. 식단 조합 생성 및 영양 랭킹
슬롯별 후보를 조합하고 식단 전체 영양값을 합산한다.
영양 균형·영양 밀도·제한 영양소·검색 유사도를 이용해 기본 랭킹을 만든다.


In [ ]:


def build_meal_combinations(slot_candidates):

    combinations = []

    for (
        staple,
        soup,
        main,
        side,
        kimchi
    ) in product(
        slot_candidates["주식"].to_dict("records"),
        slot_candidates["국"].to_dict("records"),
        slot_candidates["주찬"].to_dict("records"),
        slot_candidates["부찬"].to_dict("records"),
        slot_candidates["김치"].to_dict("records")
    ):

        menus = {
            "주식": staple,
            "국": soup,
            "주찬": main,
            "부찬": side,
            "김치": kimchi
        }

        # 같은 메뉴가 두 슬롯에 동시에 들어가는 조합 제거
        menu_codes = [
            menu["menu_fd_Code"]
            for menu in menus.values()
        ]

        if len(menu_codes) != len(set(menu_codes)):
            continue

        combination = {
            "주식": staple["menu_fd_Nm"],
            "국": soup["menu_fd_Nm"],
            "주찬": main["menu_fd_Nm"],
            "부찬": side["menu_fd_Nm"],
            "김치": kimchi["menu_fd_Nm"]
        }

        # 메뉴 코드도 보존
        for slot_name, menu in menus.items():
            combination[
                f"{slot_name}_code"
            ] = menu["menu_fd_Code"]

        # 식단 전체 영양성분 합산
        for nutrient in nutrient_cols:

            values = [
                menu.get(nutrient)
                for menu in menus.values()
            ]

            valid_values = [
                value
                for value in values
                if pd.notna(value)
            ]

            combination[nutrient] = (
                sum(valid_values)
                if valid_values
                else np.nan
            )

        # RAG 의미 유사도 평균
        combination["retrieval_score"] = np.mean([
            menu["retrieval_score"]
            for menu in menus.values()
        ])

        combinations.append(
            combination
        )

    return pd.DataFrame(
        combinations
    )


meal_combinations_df = build_meal_combinations(
    filtered_slot_candidates
)

print(
    "생성된 식단 조합 :",
    len(meal_combinations_df)
)

display(
    meal_combinations_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "retrieval_score"
        ]
    ].head()
)


In [ ]:
def range_score(
    series,
    lower,
    upper
):
    """
    적정 범위 내부 = 1
    범위 밖 = 거리에 따라 점진적 감점
    """
    score = pd.Series(
        1.0,
        index=series.index
    )

    width = upper - lower

    low_mask = series < lower
    high_mask = series > upper

    score.loc[low_mask] = (
        1
        - (
            lower
            - series.loc[low_mask]
        )
        / width
    )

    score.loc[high_mask] = (
        1
        - (
            series.loc[high_mask]
            - upper
        )
        / width
    )

    return score.clip(
        lower=0,
        upper=1
    )


def rank_meal_combinations(
    meal_df,
    nutrition_high,
    nutrition_low,
    top_k=2000
):

    result = meal_df.copy()

    # --------------------------------
    # 1. 다량영양소 에너지 비율
    # --------------------------------

    result = result[
        result["에너지"] > 0
    ].copy()

    result["carb_energy_ratio"] = (
        result["탄수화물"]
        * 4
        / result["에너지"]
        * 100
    )

    result["protein_energy_ratio"] = (
        result["단백질"]
        * 4
        / result["에너지"]
        * 100
    )

    result["fat_energy_ratio"] = (
        result["지방"]
        * 9
        / result["에너지"]
        * 100
    )

    # 2025 KDRI
    result["carb_balance_score"] = (
        range_score(
            result["carb_energy_ratio"],
            50,
            65
        )
    )

    result["protein_balance_score"] = (
        range_score(
            result["protein_energy_ratio"],
            10,
            20
        )
    )

    result["fat_balance_score"] = (
        range_score(
            result["fat_energy_ratio"],
            15,
            30
        )
    )

    result["macro_balance_score"] = (
        result[
            [
                "carb_balance_score",
                "protein_balance_score",
                "fat_balance_score"
            ]
        ]
        .mean(axis=1)
    )

    # --------------------------------
    # 2. 미량영양소 영양밀도
    #    1000 kcal 기준 상대평가
    # --------------------------------

    beneficial_cols = [
        "총 식이섬유",
        "칼슘",
        "철",
        "마그네슘",
        "칼륨",
        "아연",
        "비타민 A",
        "비타민 D",
        "비타민 C",
        "티아민",
        "리보플라빈",
        "비타민 B12",
        "비타민 K1",
        "오메가3 지방산"
    ]

    beneficial_scores = []

    for col in beneficial_cols:

        if col not in result.columns:
            continue

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        if density.notna().any():

            score_col = (
                f"_benefit_{col}"
            )

            result[score_col] = (
                density
                .rank(pct=True)
                .fillna(0.5)
            )

            beneficial_scores.append(
                score_col
            )

    if beneficial_scores:

        result[
            "micronutrient_score"
        ] = (
            result[
                beneficial_scores
            ]
            .mean(axis=1)
        )

    else:

        result[
            "micronutrient_score"
        ] = 0.5

    # --------------------------------
    # 3. 제한 영양소
    # --------------------------------

    limit_cols = [
        "당류",
        "나트륨",
        "총 포화 지방산",
        "총 트랜스 지방산",
        "콜레스테롤"
    ]

    limit_scores = []

    for col in limit_cols:

        if col not in result.columns:
            continue

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        if density.notna().any():

            score_col = (
                f"_limit_{col}"
            )

            result[score_col] = (
                1
                - density.rank(
                    pct=True
                )
            ).fillna(0.5)

            limit_scores.append(
                score_col
            )

    if limit_scores:

        result[
            "limit_nutrient_score"
        ] = (
            result[
                limit_scores
            ]
            .mean(axis=1)
        )

    else:

        result[
            "limit_nutrient_score"
        ] = 0.5

    # --------------------------------
    # 4. 에너지 극단값 억제
    # 절대 kcal 기준을 임의로 만들지 않고
    # 현재 후보군의 중앙값 근처를 선호
    # --------------------------------

    energy_pct = (
        result["에너지"]
        .rank(pct=True)
    )

    result[
        "energy_center_score"
    ] = (
        1
        - (
            energy_pct
            - 0.5
        ).abs()
        * 2
    ).clip(
        lower=0,
        upper=1
    )

    # --------------------------------
    # 5. 기본 식단 영양 품질
    # --------------------------------

    result[
        "baseline_nutrition_score"
    ] = (
        result[
            [
                "macro_balance_score",
                "micronutrient_score",
                "limit_nutrient_score",
                "energy_center_score"
            ]
        ]
        .mean(axis=1)
    )

    # --------------------------------
    # 6. 사용자 요청
    # --------------------------------

    preference_scores = []

    for col in nutrition_high:

        if col not in result.columns:
            continue

        score_col = (
            f"_pref_high_{col}"
        )

        if col == "단백질":

            # 단백질을 높게 원하더라도
            # 20%를 넘어 무한 보상하지 않음
            result[score_col] = (
                result[
                    "protein_energy_ratio"
                ]
                .clip(
                    lower=0,
                    upper=20
                )
                / 20
            )

        else:

            density = (
                result[col]
                / result["에너지"]
                * 1000
            )

            result[score_col] = (
                density.rank(
                    pct=True
                )
                .fillna(0.5)
            )

        preference_scores.append(
            score_col
        )

    for col in nutrition_low:

        if col not in result.columns:
            continue

        score_col = (
            f"_pref_low_{col}"
        )

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        result[score_col] = (
            1
            - density.rank(
                pct=True
            )
        ).fillna(0.5)

        preference_scores.append(
            score_col
        )

    if preference_scores:

        result[
            "preference_score"
        ] = (
            result[
                preference_scores
            ]
            .mean(axis=1)
        )

        # 기본 영양 균형과 사용자 요구를
        # 동일 비중으로 평가
        result[
            "nutrition_score"
        ] = (
            result[
                [
                    "baseline_nutrition_score",
                    "preference_score"
                ]
            ]
            .mean(axis=1)
        )

    else:

        result[
            "preference_score"
        ] = 0.5

        result[
            "nutrition_score"
        ] = result[
            "baseline_nutrition_score"
        ]

    # --------------------------------
    # 7. 최종 정렬
    # --------------------------------

    return (
        result
        .sort_values(
            [
                "nutrition_score",
                "retrieval_score"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
ranked_meal_df = rank_meal_combinations(
    meal_combinations_df,
    nutrition_high=conditions["nutrition_high"],
    nutrition_low=conditions["nutrition_low"],
    top_k=2000
)

display(
    ranked_meal_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "nutrition_score",
            "retrieval_score"
        ]
    ].head(10)
)

# Part 9. ML 영양 군집 적용
K 검증 결과를 바탕으로 최종 K=6을 사용한다.
메뉴별 영양 Cluster를 생성하고 식단 내 Cluster 다양성을 보조 점수(5%)로 반영한다.


In [ ]:
from menu_cluster_model import MenuNutritionCluster


In [ ]:

# ML 모델 학습
cluster_model = MenuNutritionCluster(
    k_min=6,
    k_max=6,
    random_state=42
)

clustered_menu_df = cluster_model.fit_predict(
    recommendable_menu_df
)

# 메뉴 코드 → 영양 군집
cluster_map = (
    clustered_menu_df
    .set_index("menu_fd_Code")["nutrition_cluster"]
    .to_dict()
)

print("선택된 Cluster 수 :", cluster_model.best_k_)
print(
    "Silhouette Score :",
    round(cluster_model.silhouette_score_, 4)
)

display(cluster_model.cluster_search_)
display(
    cluster_model.cluster_summary(
        clustered_menu_df
    )
)


In [ ]:
cluster_counts = (
    clustered_menu_df["nutrition_cluster"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.title("영양 군집별 메뉴 수")
plt.xlabel("Cluster")
plt.ylabel("메뉴 수")

for bar, count in zip(
    bars,
    cluster_counts.values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"{count:,}",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
cluster_summary_df = (
    cluster_model.cluster_summary(
        clustered_menu_df
    )
)

heatmap_features = [
    "carb_energy_ratio",
    "protein_energy_ratio",
    "fat_energy_ratio",
    "총 식이섬유_density",
    "나트륨_density",
    "칼슘_density",
    "철_density",
    "마그네슘_density",
    "칼륨_density",
    "아연_density"
]

cluster_heatmap = (
    cluster_summary_df
    .set_index("nutrition_cluster")[heatmap_features]
)

# 서로 단위가 다르므로 표준화
cluster_heatmap_scaled = (
    cluster_heatmap - cluster_heatmap.mean()
) / cluster_heatmap.std()

plt.figure(figsize=(12, 6))

im = plt.imshow(
    cluster_heatmap_scaled,
    cmap="coolwarm",
    aspect="auto"
)

plt.colorbar(
    im,
    label="표준화 값"
)

plt.xticks(
    range(len(heatmap_features)),
    heatmap_features,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(cluster_heatmap_scaled.index)),
    cluster_heatmap_scaled.index
)

# 셀 안에 표준화 값 표시
for i in range(cluster_heatmap_scaled.shape[0]):
    for j in range(cluster_heatmap_scaled.shape[1]):
        value = cluster_heatmap_scaled.iloc[i, j]

        if pd.notna(value):
            plt.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

plt.xlabel("영양 특성")
plt.ylabel("Cluster")
plt.title("Cluster별 영양 특성 Heatmap")

plt.tight_layout()
plt.show()

In [ ]:
ML_SLOTS = [
    "주식",
    "국",
    "주찬",
    "부찬",
    "김치"
]

ml_ranked_meal_df = ranked_meal_df.copy()

# 각 메뉴가 속한 영양 Cluster 연결
cluster_cols = []

for slot in ML_SLOTS:

    cluster_col = f"{slot}_cluster"

    ml_ranked_meal_df[cluster_col] = (
        ml_ranked_meal_df[
            f"{slot}_code"
        ]
        .map(cluster_map)
    )

    cluster_cols.append(
        cluster_col
    )


# 한 식단 안에 서로 다른 영양 Cluster가
# 얼마나 포함되는지 계산
ml_ranked_meal_df[
    "cluster_diversity_score"
] = (
    ml_ranked_meal_df[
        cluster_cols
    ]
    .nunique(axis=1)
    / len(ML_SLOTS)
)


# 기존 영양점수를 핵심으로 유지하고
# ML 영양 다양성을 보조적으로 반영
ml_ranked_meal_df[
    "final_score"
] = (
    0.95
    * ml_ranked_meal_df[
        "nutrition_score"
    ]
    +
    0.05
    * ml_ranked_meal_df[
        "cluster_diversity_score"
    ]
)


ml_ranked_meal_df = (
    ml_ranked_meal_df
    .sort_values(
        [
            "final_score",
            "nutrition_score",
            "retrieval_score"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

# Part 10. 최종 추천 식단 선택
ML 보정 후 점수 순으로 정렬한 뒤, 동일 메뉴 구성이 과도하게 반복되지 않도록 서로 다른 식단 3개를 선택한다.


In [ ]:
# 다양한 식단 선택
MEAL_SLOTS = [
    "주식",
    "국",
    "주찬",
    "부찬",
    "김치"
]


def select_diverse_meals(
    ranked_df,
    n_meals=3,
    max_shared_slots=2
):
    selected = []

    for _, candidate in ranked_df.iterrows():

        # 첫 번째는 최고 점수 식단 그대로 선택
        if not selected:
            selected.append(candidate)
            continue

        is_diverse = True

        for chosen in selected:

            shared_slots = sum(
                candidate[slot] == chosen[slot]
                for slot in MEAL_SLOTS
            )

            # 기존 선택 식단과 너무 많이 겹치면 제외
            if shared_slots > max_shared_slots:
                is_diverse = False
                break

        if is_diverse:
            selected.append(candidate)

        if len(selected) >= n_meals:
            break

    return pd.DataFrame(selected).reset_index(drop=True)

In [ ]:
diverse_meal_df = select_diverse_meals(
    ml_ranked_meal_df,
    n_meals=3,
    max_shared_slots=1
)

display(
    diverse_meal_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "nutrition_score",
            "cluster_diversity_score",
            "final_score",
            "retrieval_score"
        ]
    ]
)

# Part 11. 가이드라인 RAG 및 결과 검증
공식 가이드라인 문서를 별도 RAG로 검색해 추천 근거를 확인한다.
이 구간은 현재 랭킹 점수를 직접 변경하는 단계가 아니라 근거 검색·검증 단계다.


## 가이드라인 RAG

In [ ]:
# Guideline corpus
guideline_records = [
    {
        "doc_id": "KDRI_PROTEIN_RATIO",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "단백질",
        "document": """
대상: 3세 이상
영양소: 단백질
기준: 단백질 에너지적정비율은 총 에너지 섭취량의 10~20%이다.
활용: 식사의 단백질 적정성을 평가할 때 단순히 단백질 g이 높을수록 좋다고 판단하지 않고,
총 에너지 중 단백질이 차지하는 비율을 함께 고려한다.
""".strip()
    },

    {
        "doc_id": "KDRI_PROTEIN_OLDER",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "노인 단백질",
        "document": """
대상: 65세 이상 성인
영양소: 단백질
65~74세 남성 권장섭취량은 60 g/일이다.
75세 이상 남성 권장섭취량은 60 g/일이다.
65~74세 여성 권장섭취량은 50 g/일이다.
75세 이상 여성 권장섭취량은 50 g/일이다.
성별이 확인되지 않은 경우 특정 한 값을 개인 기준으로 확정하지 않는다.
""".strip()
    },

    {
        "doc_id": "KDRI_SODIUM_OLDER",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "노인 나트륨",
        "document": """
대상: 65세 이상 성인
영양소: 나트륨
65~74세 충분섭취량은 1,300 mg/일이고,
만성질환위험감소섭취량은 1,900 mg/일이다.
75세 이상 충분섭취량은 1,200 mg/일이고,
만성질환위험감소섭취량은 1,800 mg/일이다.
연령이 명확하지 않은 경우 65~74세와 75세 이상 기준의 차이를 고려한다.
""".strip()
    },

    {
        "doc_id": "KSH_HYPERTENSION_SODIUM",
        "source": "대한고혈압학회, 2026 제6판 고혈압 진료지침",
        "topic": "고혈압",
        "document": """
질환: 고혈압
비약물적 생활요법으로 나트륨 섭취 제한이 강하게 권고된다.
체중 조절, 절주, 금연, 규칙적인 신체활동,
건강한 식사와 함께 포괄적인 생활습관 개선을 권고한다.
고혈압이라는 질환명만으로 임의의 음식이나 영양 수치를 생성하지 않고
근거가 있는 영양 기준과 함께 적용한다.
""".strip()
    }
]

guideline_documents_df = pd.DataFrame(
    guideline_records
)

display(guideline_documents_df)

In [ ]:
# Guideline 임베딩
guideline_texts = (
    guideline_documents_df[
        "document"
    ]
    .tolist()
)

guideline_output = bge_model.encode(
    guideline_texts,
    batch_size=4,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

guideline_embeddings = np.asarray(
    guideline_output["dense_vecs"],
    dtype="float32"
)

faiss.normalize_L2(
    guideline_embeddings
)

print(
    "Guideline embedding shape :",
    guideline_embeddings.shape
)

In [ ]:
# Guideline FAISS
guideline_index = faiss.IndexFlatIP(
    guideline_embeddings.shape[1]
)

guideline_index.add(
    guideline_embeddings
)

print(
    "Guideline documents :",
    guideline_index.ntotal
)

In [ ]:
# Guideline 검색 함수
def retrieve_guidelines(
    conditions,
    top_k=4
):

    query_parts = []

    if conditions.get("target"):
        query_parts.append(
            f"급식 대상: {conditions['target']}"
        )

    if conditions.get("diseases"):
        query_parts.append(
            "건강 상태: "
            + ", ".join(
                conditions["diseases"]
            )
        )

    if conditions.get("nutrition_high"):
        query_parts.append(
            "많이 원하는 영양소: "
            + ", ".join(
                conditions["nutrition_high"]
            )
        )

    if conditions.get("nutrition_low"):
        query_parts.append(
            "적게 원하는 영양소: "
            + ", ".join(
                conditions["nutrition_low"]
            )
        )

    if conditions.get("keywords"):
        query_parts.append(
            "추가 조건: "
            + ", ".join(
                conditions["keywords"]
            )
        )

    query = "\n".join(
        query_parts
    )

    output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(
        query_vector
    )

    search_k = min(
        top_k,
        guideline_index.ntotal
    )

    scores, indices = (
        guideline_index.search(
            query_vector,
            search_k
        )
    )

    result = (
        guideline_documents_df
        .iloc[indices[0]]
        .copy()
    )

    result["retrieval_score"] = (
        scores[0]
    )

    return result.reset_index(
        drop=True
    )

In [ ]:
# 현재 조건으로 검색
retrieved_guidelines_df = (
    retrieve_guidelines(
        conditions,
        top_k=4
    )
)

display(
    retrieved_guidelines_df[
        [
            "doc_id",
            "topic",
            "source",
            "retrieval_score"
        ]
    ]
)

In [ ]:
# 식단의 단백질 에너지비율 계산
def add_guideline_features(meal_df):
    result = meal_df.copy()

    # 단백질 1g = 4 kcal
    result["protein_energy_ratio"] = np.where(
        (
            result["에너지"].notna()
            & result["단백질"].notna()
            & (result["에너지"] > 0)
        ),
        (
            result["단백질"]
            * 4
            / result["에너지"]
            * 100
        ),
        np.nan
    )

    return result

In [ ]:
meal_combinations_guideline_df = (
    add_guideline_features(
        meal_combinations_df
    )
)

display(
    meal_combinations_guideline_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "에너지",
            "단백질",
            "protein_energy_ratio",
            "나트륨"
        ]
    ].head()
)

In [ ]:
def inspect_meal_components(
    meal_row
):
    records = []

    for slot in [
        "주식",
        "국",
        "주찬",
        "부찬",
        "김치"
    ]:
        code = meal_row[
            f"{slot}_code"
        ]

        menu_row = (
            recommendable_menu_df[
                recommendable_menu_df[
                    "menu_fd_Code"
                ] == code
            ]
            .iloc[0]
        )

        weight_row = (
            menu_df[
                menu_df["fd_Code"]
                == code
            ]
            .iloc[0]
        )

        records.append({
            "슬롯": slot,
            "메뉴": menu_row["menu_fd_Nm"],
            "메뉴중량_g": weight_row["fd_Wgh"],
            "에너지_kcal": menu_row["에너지"],
            "탄수화물_g": menu_row["탄수화물"],
            "단백질_g": menu_row["단백질"],
            "지방_g": menu_row["지방"],
            "나트륨_mg": menu_row["나트륨"]
        })

    return pd.DataFrame(records)

In [ ]:
meal_check_df = inspect_meal_components(
    meal_combinations_guideline_df.iloc[0]
)

display(meal_check_df)

print(
    "\n총 에너지 :",
    meal_check_df["에너지_kcal"].sum()
)

print(
    "총 메뉴 중량 :",
    meal_check_df["메뉴중량_g"].sum()
)

# Appendix. K 선택 검증 기록
최종 파이프라인은 K=6으로 고정한다.
아래 셀은 K=5·6·7을 비교했던 모델 선택 근거를 재현할 때만 실행한다.


In [ ]:
from menu_cluster_model import MenuNutritionCluster

validation_rows = []

for k in [5, 6, 7]:
    model = MenuNutritionCluster(
        k_min=k,
        k_max=k,
        random_state=42
    )

    clustered = model.fit_predict(
        recommendable_menu_df
    )

    cluster_sizes = (
        clustered["nutrition_cluster"]
        .value_counts()
        .sort_values()
        .tolist()
    )

    validation_rows.append({
        "k": k,
        "silhouette_score": model.silhouette_score_,
        "min_cluster_size": min(cluster_sizes),
        "max_cluster_size": max(cluster_sizes),
        "cluster_sizes": cluster_sizes
    })

cluster_validation_df = pd.DataFrame(validation_rows)

display(cluster_validation_df)


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    cluster_validation_df["k"],
    cluster_validation_df["silhouette_score"],
    marker="o"
)

plt.title("K별 Silhouette Score 비교")
plt.xlabel("Cluster 수 (K)")
plt.ylabel("Silhouette Score")
plt.xticks(cluster_validation_df["k"])

plt.tight_layout()
plt.show()
